<a href="https://colab.research.google.com/github/rafahcs/Projeto_AF/blob/main/projetoaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. INTRODUÇÃO
− Linguagem escolhida
− Motivação

* O problema consiste em um autômato finito determinístico(AFD) que reconheça a linguagaem formada pelas notações dos elementos pertencentes aos conjuntos dos números inteiros, números reais, números hexadecimais, números em notação científica e números complexos.

* O AFD recebe uma fita de tamanho variável. Cada estado armazena o último símbolo anterior lido após a transição do estado antecesssor.

* Uma palavra só é aceita se o útimo símbolo for lido por um estado que seja final. Palavras não aceitas são rejeitadas e o autômato libera o armazenamento da fita e lê a próxima palavra.

* A linguagem escolhida foi L(A) = { w | w é uma palavra real ou inteira ou complexa ou notação científica ou hexadecimal}

* A estratégia adotada foi a de execução da fita por cada autômato representado por uma função builder_ que percorre a fita lendo cada símbolo e comparando com o valor esperado para o alfabeto do autômato. Ao final são rodados os testes unitários que retornam se a fita tem uma palavra aceita ou não na linguagem.


2. DEFINIÇÃO FORMAL DO AF
− 5−upla completa
− Diagrama de estados (opcional)

Q: Um conjunto finito de estados.

Σ: Um alfabeto finito de símbolos (alfabeto da fita).

δ: Função de transição

qO: Estado inicial

F: Conjunto de estados finais ou de aceitação

M = {Q, Σ, δ, qO, F}

Q = {q0, q1, q2, q3, q4, q5, q6, q7}

Σ = $\mathbb{Z}$ $\cup$ $\mathbb{R}$ $\cup$ {A,B,C,D,E,F} $\cup$ $\mathbb{C}$

δ = Q x Σ $\rightarrow$ Q

q0 = {q0}

F = {q2, q4, q5, q6, q7}


3. IMPLEMENTAÇÃO
− Código completo da classe FiniteAutomata
− Funções auxiliares


In [3]:
from typing import Dict, Tuple


class FiniteAutomata:
    delta: Dict[Tuple[str, str], str] = None

#inicialização do automato
    def __init__(self, tape, initial_state, final_states, delta):
        self.tape = list(tape)  #cria fita
        self.blank = "_"
        self.head = 0 #cabeça de leitura da fita

        self.initial_state = initial_state  #estado inicial
        self.current_state = initial_state  #estado atual
        self.final_states = final_states  #estados finais
        self.delta = delta  #estados de aceitação

#leitura de símbolo
    def read_symbol(self):
        if self.head >= len(self.tape):
            return self.blank
        return self.tape[self.head]

#função de transição
    def step(self):
        symbol = self.read_symbol()
        key = (self.current_state, symbol)

        if key not in self.delta:
            return False

        self.current_state = self.delta[key]
        self.head += 1
        return True

    def execute(self):
        while self.head < len(self.tape):
            if not self.step():
                return False
        return self.current_state in self.final_states


# INTEIRO
def build_integer_af(tape):
    delta = {}
    digits = "0123456789"

    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"

    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    return FiniteAutomata(tape, "q0", {"q2"}, delta)


# REAL
def build_real_af(tape):
    delta = {}
    digits = "0123456789"

    # início
    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", ".")] = "q3"

    # número inteiro inicial
    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    # ponto depois de número → aceita "3."
    delta[("q2", ".")] = "q5"

    # ponto sem número antes → precisa de dígito depois
    for d in digits:
        delta[("q3", d)] = "q4"

    # parte decimal normal
    for d in digits:
        delta[("q4", d)] = "q4"
        delta[("q5", d)] = "q4"

    # sinal seguido de ponto
    delta[("q1", ".")] = "q3"

    return FiniteAutomata(
        tape,
        "q0",
        {"q2", "q4", "q5"},  # <-- chave da correção
        delta
    )


# CIENTÍFICA
def build_scientific_af(tape):
    delta = {}
    digits = "0123456789"

    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", ".")] = "q3"

    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"
        delta[("q3", d)] = "q4"
        delta[("q4", d)] = "q4"

    delta[("q1", ".")] = "q3"
    delta[("q2", ".")] = "q3"
    delta[("q3", "e")] = "q5"
    delta[("q3", "E")] = "q5"
    delta[("q2", "e")] = "q5"
    delta[("q2", "E")] = "q5"
    delta[("q4", "e")] = "q5"
    delta[("q4", "E")] = "q5"

    delta[("q5", "+")] = "q6"
    delta[("q5", "-")] = "q6"

    for d in digits:
        delta[("q5", d)] = "q7"
        delta[("q6", d)] = "q7"
        delta[("q7", d)] = "q7"

    return FiniteAutomata(tape, "q0", {"q7"}, delta)


# HEXADECIMAL
def build_hexadecimal_af(tape):
    delta = {}
    hex_digits = "0123456789abcdefABCDEF"

    # 0x...
    delta[("q0", "0")] = "q1"
    delta[("q1", "x")] = "q2"
    delta[("q1", "X")] = "q2"

    # #ABC
    delta[("q0", "#")] = "q3"

    for d in hex_digits:
        if d != "0": # Se for 0, já definimos que vai para q1
            delta[("q0", d)] = "q4"
        delta[("q1", d)] = "q4" # Começando direto (ex: 1A3)
        delta[("q2", d)] = "q4"
        delta[("q3", d)] = "q4"
        delta[("q4", d)] = "q4"

    delta[("q4", "h")] = "q5"
    delta[("q4", "H")] = "q5"

    return FiniteAutomata(tape, "q0", {"q4", "q5"}, delta)


# COMPLEXO
def build_complex_af(tape):
    delta = {}
    digits = "0123456789"

    # --- INÍCIO ---
    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", "i")] = "q6"

    # número inicial (parte real OU imaginária direta)
    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    # ponto decimal (opcional)
    delta[("q2", ".")] = "q3"
    for d in digits:
        delta[("q3", d)] = "q3"

    # número seguido de i → forma "bi"
    delta[("q2", "i")] = "q6"
    delta[("q3", "i")] = "q6"

    # parte real seguida de + ou -
    delta[("q2", "+")] = "q4"
    delta[("q2", "-")] = "q4"
    delta[("q3", "+")] = "q4"
    delta[("q3", "-")] = "q4"

    # após sinal da parte imaginária:
    delta[("q4", "i")] = "q6"  # aceita "3+i"

    for d in digits:
        delta[("q4", d)] = "q5"

    # parte imaginária numérica
    for d in digits:
        delta[("q5", d)] = "q5"

    delta[("q5", ".")] = "q4"
    delta[("q5", "i")] = "q6"

    return FiniteAutomata( tape, "q0", {"q6"}, delta )


# construtor dos automatos
automata = {
        "Inteiro": build_integer_af,
        "Real": build_real_af,
        "Cientifico": build_scientific_af,
        "Hexadecimal": build_hexadecimal_af,
        "Complexo": build_complex_af
    }


# TESTES
def run_tests():
    tests = {
        "Inteiro": [
            ("1", True), ("+2", True), ("-3", True),                                # Pequenas/Óbvias
            ("", False), (" ", False), ("+", False), ("-", False), ("+-12", False), # Malformadas
            ("13c", False), ("++10", False), ("+12bc", False),                      # Inválidas
        ],
        "Real": [
            ("3.14", True), ("-0.5", True), ("+2.0", True), (".5", True), ("5.", True), # Óbvias
            ("1.2.3", False), ("+-2.5", False), (".", False)                            # Malformadas
            ("e.5", False), ("1,5", False)                                              # Inválidas
        ],
        "Cientifico": [
            ("1e0", True), ("6E+7", True), ("-5e-10", True),  # Óbvias
            ("e10", False), ("1e", False), ("1e+2", False),   # Malformadas
            ("1.5e3.2", False), ("2 e 2", False),             # Inválidas
        ],
        "Hexadecimal": [
            ("0xF", True), ("#1", True), ("Fh", True),        # Óbvias
            ("0x", False), ("#", False), ("h", False),        # Malformadas
            ("0xG1", False), ("#ZZ", False), ("1A3Xh", False) # Inválidas

        ],
        "Complexo": [
            ("i", True), ("+i", True), ("-i", True),            # Pequenas
            ("3+4i", True), ("2.5-1.3i", True),                 # Óbvias
            ("2--i", True), ("i+3", True), ("6+4j", True),      # Malformadas
            ("2.5.3i", True), ("2.5.3+i", True), ("+-i", True)  # Inválidas
        ]
    }

    for name, builder in automata.items():
        print(f"\n=== {name} ===")
        for t, expected in tests.get(name, []):
            result = builder(t).execute()

            if result == expected:
                print(f"OK   {t}")
            else:
                print(f"FAIL {t} esperado {expected}, obteve {result}")


# EXECUÇÃO
if __name__ == "__main__":
    run_tests()

4. TESTES
− Casos pequenos
− Verificação de correção

    1. Instâncias pequenas;
    2. Instâncias inválidas malformadas;
    3. Casos óbvios de aceitação/rejeição.


In [4]:
# TESTES
def run_tests():
    tests = {
        "Inteiro": [
            ("123", True),
            ("+12", True),
            ("-10", True),
            ("+abc", False) #Não aceita
        ],
        "Real": [
            ("3.14", True),
            ("-0.5", True),
            ("+2.0", True),
            ("5.", True),
            (".5", True),
            ("..5", False)  #Não aceita
        ],
        "Cientifico": [
            ("1.23e4", True),
            ("-5E-10", True),
            ("6E+7", True),
            ("5e3", True),
            ("2.e-2", True),
            ("1e", False), #Não aceita
            ("1e+", False), #Não aceita
            ("e10", False)  #Não aceita
        ],
        "Hexadecimal": [
            ("0xFF", True),
            ("0x1A3", True),
            ("#ABC", True),
            ("1A3h", True),
            ("0x", False) #Não aceita
        ],
        "Complexo": [
            ("2", False), #Não aceita
            ("3+4i", True),
            ("2-10i", True),
            ("3+i", True),
            ("+4i", True),
            ("2.5-1.3i", True)
        ]
    }

    for name, builder in automata.items():
        print(f"\n=== {name} ===")
        for t, expected in tests.get(name, []):
            result = builder(t).execute()

            if result == expected:
                print(f"OK   {t}")
            else:
                print(f"FAIL {t} esperado {expected}, obteve {result}")

5. DEMONSTRAÇÃO FORMAL
− Prova de corretude do AF apresentado


6. CONCLUSÃO
− Principais aprendizados
− Limitações do estudo

*   Um aprendizado do projeto do AFD é que ..
*   A limitação é que a implementação do autômato fois feita para ser executada em uma máquina com  

    - 107.7 GB de disco
    - 12.7 GB de RAM
    - ... núcleos
    - registradores de ... bits